# Agricultural Subsidy Analysis — CAP Payments by MSA

DEFRA / RPA Common Agricultural Policy (CAP) payments data, 2024.
Source: *2024 All CAP Search Results Data P14*, sheets RPA_1 & RPA2 (England).

Payments mapped to MSAs via ONSPD postcode district → modal LAD → CA/DPP lookup.
41% of rows are in non-MSA areas (counties without devolution deals) — expected, not data loss.

In [ ]:
import pandas as pd
import altair as alt
import numpy as np

BASE = "/Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Documents/GitHub/RADataHub/msa_map"

TEAL  = "#36b7b4"
NAVY  = "#122b39"
BLUE  = "#179fdb"
BACKGROUND = "white"

TIER_COLORS = {
    "Established": NAVY,
    "Existing":    BLUE,
    "DPP":         "#a8c0de",
}
YNYCA_NAME = "York & North Yorkshire"

alt.data_transformers.enable("default", max_rows=None)

## 1. Data loading and pipeline

In [ ]:
# ── CAP payments ─────────────────────────────────────────────────────────
CAP_PATH = f"{BASE}/data/raw/agri/2024_All_CAP_Search_Results_Data_P14.xlsx"
PC_LOOKUP = f"{BASE}/data/processed/postcode_district_msa_lookup.csv"

keep_cols = ["PostcodePrefix_F202B", "Total", "DirectEAGFTotal", "RuralDevelopmentTotal"]
cap1 = pd.read_excel(CAP_PATH, sheet_name="RPA_1", usecols=keep_cols)
cap2 = pd.read_excel(CAP_PATH, sheet_name="RPA2",  usecols=keep_cols)
cap  = pd.concat([cap1, cap2], ignore_index=True)
cap["district"] = cap["PostcodePrefix_F202B"].astype(str).str.strip().str.upper()

pc_msa = pd.read_csv(PC_LOOKUP)[["district", "msa_name"]]
cap = cap.merge(pc_msa, on="district", how="left")

cap_agg = (
    cap[cap["msa_name"].notna()]
    .groupby("msa_name")
    .agg(
        n_recipients     = ("Total", "count"),
        total_m          = ("Total",              lambda x: x.sum() / 1e6),
        direct_m         = ("DirectEAGFTotal",     lambda x: x.sum() / 1e6),
        rural_dev_m      = ("RuralDevelopmentTotal",lambda x: x.sum() / 1e6),
    )
    .round(2)
    .reset_index()
)

# ── Population, area, tier ────────────────────────────────────────────────
pop = pd.read_csv(f"{BASE}/data/processed/msa_population_ranks.csv",
                  usecols=["msa_name", "tier", "area_km2", "pop_2023"])

# ── Rurality scores ───────────────────────────────────────────────────────
NAME_FIX = {
    "York and North Yorkshire":        "York & North Yorkshire",
    "Hull and East Yorkshire":         "Hull & East Yorkshire",
    "Cambridgeshire and Peterborough": "Cambridgeshire & Peterborough",
}
rur = pd.read_csv(f"{BASE}/data/processed/msa_rural_urban_scores.csv",
                  usecols=["msa_name", "pop_weighted_rurality_score", "pct_pop_rural"])
rur["msa_name"] = rur["msa_name"].replace(NAME_FIX)

# ── Merge ─────────────────────────────────────────────────────────────────
cap_msa = (cap_agg
           .merge(pop, on="msa_name", how="left")
           .merge(rur, on="msa_name", how="left"))

cap_msa["gbp_per_km2"]      = (cap_msa["total_m"] * 1e6 / cap_msa["area_km2"]).round(0)
cap_msa["gbp_per_capita"]   = (cap_msa["total_m"] * 1e6 / cap_msa["pop_2023"]).round(2)
cap_msa["pct_rural_dev"]    = (cap_msa["rural_dev_m"] / cap_msa["total_m"] * 100).round(1)
cap_msa["_hl"]              = cap_msa["msa_name"] == YNYCA_NAME
cap_msa["tier"]             = cap_msa["tier"].fillna("DPP")  # Devon & Torbay etc.
cap_msa["dot_color"]        = cap_msa.apply(
    lambda r: TEAL if r["_hl"] else TIER_COLORS.get(r["tier"], "#a8c0de"), axis=1
)

print(cap_msa[["msa_name","tier","total_m","direct_m","rural_dev_m",
               "gbp_per_km2","gbp_per_capita","pop_weighted_rurality_score"]]
      .sort_values("total_m", ascending=False).to_string(index=False))

## 2. Chart A — Total CAP payments by MSA

In [ ]:
sort_a = cap_msa.sort_values("total_m", ascending=False)["msa_name"].tolist()

bars_a = alt.Chart(cap_msa).mark_bar(cornerRadiusEnd=3, height=14).encode(
    y=alt.Y("msa_name:N", sort=sort_a, title=None,
            axis=alt.Axis(labelFontSize=10, grid=False)),
    x=alt.X("total_m:Q", title="Total CAP payments (£m, 2024)",
            axis=alt.Axis(grid=False)),
    color=alt.Color("dot_color:N", scale=None),
    tooltip=[
        alt.Tooltip("msa_name:N",    title="MSA"),
        alt.Tooltip("tier:N",        title="Tier"),
        alt.Tooltip("total_m:Q",     title="Total (£m)",      format=".1f"),
        alt.Tooltip("direct_m:Q",    title="Direct (£m)",     format=".1f"),
        alt.Tooltip("rural_dev_m:Q", title="Rural dev (£m)",  format=".1f"),
        alt.Tooltip("n_recipients:Q",title="Recipients",      format=","),
    ],
)
# Value labels
labels_a = alt.Chart(cap_msa).mark_text(
    align="left", dx=4, fontSize=9, color=NAVY,
).encode(
    y=alt.Y("msa_name:N", sort=sort_a),
    x=alt.X("total_m:Q"),
    text=alt.Text("total_m:Q", format=".1f"),
)

chart_a = (bars_a + labels_a).properties(
    width=420, height=420,
    title=alt.TitleParams(
        text="Total CAP payments by MSA, 2024",
        subtitle=[
            "DEFRA/RPA CAP payments mapped via postcode district → LAD → MSA.",
            "Colour: dark = Established CA, blue = Existing CA, light = DPP.  YNYCA in teal.",
        ],
        fontSize=13, subtitleFontSize=9, anchor="start",
    ),
).configure_view(strokeWidth=0).configure(background=BACKGROUND).configure_axis(grid=False)

chart_a

## 3. Chart B — CAP payments per km² (subsidy density)

In [ ]:
sort_b = cap_msa.sort_values("gbp_per_km2", ascending=False)["msa_name"].tolist()

bars_b = alt.Chart(cap_msa).mark_bar(cornerRadiusEnd=3, height=14).encode(
    y=alt.Y("msa_name:N", sort=sort_b, title=None,
            axis=alt.Axis(labelFontSize=10, grid=False)),
    x=alt.X("gbp_per_km2:Q", title="CAP payments per km² (£, 2024)",
            axis=alt.Axis(grid=False)),
    color=alt.Color("dot_color:N", scale=None),
    tooltip=[
        alt.Tooltip("msa_name:N",      title="MSA"),
        alt.Tooltip("gbp_per_km2:Q",   title="£ per km²",  format=",.0f"),
        alt.Tooltip("total_m:Q",       title="Total (£m)", format=".1f"),
        alt.Tooltip("area_km2:Q",      title="Area (km²)", format=",.0f"),
    ],
)
labels_b = alt.Chart(cap_msa).mark_text(
    align="left", dx=4, fontSize=9, color=NAVY,
).encode(
    y=alt.Y("msa_name:N", sort=sort_b),
    x=alt.X("gbp_per_km2:Q"),
    text=alt.Text("gbp_per_km2:Q", format=",.0f"),
)

chart_b = (bars_b + labels_b).properties(
    width=420, height=420,
    title=alt.TitleParams(
        text="CAP subsidy density (£ per km²) by MSA, 2024",
        subtitle="Controls for MSA size — larger geographies receive more simply because they cover more land.",
        fontSize=13, subtitleFontSize=9, anchor="start",
    ),
).configure_view(strokeWidth=0).configure(background=BACKGROUND).configure_axis(grid=False)

chart_b

## 4. Chart C — Rurality score vs CAP payments per km²

In [ ]:
from scipy.stats import pearsonr, spearmanr

cdf = cap_msa.dropna(subset=["pop_weighted_rurality_score","gbp_per_km2"])
r_p, p_p = pearsonr(cdf["pop_weighted_rurality_score"], cdf["gbp_per_km2"])
r_s, p_s = spearmanr(cdf["pop_weighted_rurality_score"], cdf["gbp_per_km2"])
print(f"Pearson r={r_p:.3f} p={p_p:.3f}  |  Spearman rho={r_s:.3f} p={p_s:.3f}")

dots_c = alt.Chart(cap_msa).mark_circle(size=80, opacity=0.85).encode(
    x=alt.X("pop_weighted_rurality_score:Q",
            title="Rurality index (pop-weighted RUC score)",
            axis=alt.Axis(grid=False)),
    y=alt.Y("gbp_per_km2:Q",
            title="CAP payments per km² (£)",
            axis=alt.Axis(grid=False)),
    color=alt.Color("dot_color:N", scale=None),
    tooltip=[
        alt.Tooltip("msa_name:N",                   title="MSA"),
        alt.Tooltip("pop_weighted_rurality_score:Q", title="Rurality index", format=".1f"),
        alt.Tooltip("gbp_per_km2:Q",                title="£ per km²",      format=",.0f"),
        alt.Tooltip("tier:N",                        title="Tier"),
    ],
)
labels_c = alt.Chart(cap_msa).mark_text(
    align="left", dx=7, fontSize=8.5,
).encode(
    x=alt.X("pop_weighted_rurality_score:Q"),
    y=alt.Y("gbp_per_km2:Q"),
    text="msa_name:N",
    color=alt.Color("dot_color:N", scale=None),
    fontWeight=alt.condition("datum._hl", alt.value("bold"), alt.value("normal")),
)
trend_c = dots_c.transform_regression(
    "pop_weighted_rurality_score", "gbp_per_km2"
).mark_line(strokeDash=[4, 3], color="#bbbbbb", strokeWidth=1.2)

chart_c = (trend_c + dots_c + labels_c).properties(
    width=480, height=360,
    title=alt.TitleParams(
        text="More rural MSAs attract more agricultural subsidy per km²",
        subtitle=f"Pearson r = {r_p:.2f} (p = {p_p:.3f}).  Trend line: OLS fit.",
        fontSize=13, subtitleFontSize=9, anchor="start",
    ),
).configure_view(strokeWidth=0).configure(background=BACKGROUND).configure_axis(grid=False)

chart_c

## 5. Chart D — Direct vs Rural Development payment split

In [ ]:
# Long-form for stacked bar
long_d = cap_msa.melt(
    id_vars=["msa_name", "dot_color"],
    value_vars=["direct_m", "rural_dev_m"],
    var_name="payment_type", value_name="amount_m"
)
long_d["payment_type"] = long_d["payment_type"].map({
    "direct_m":    "Direct (BPS, Greening, etc.)",
    "rural_dev_m": "Rural Development",
})

sort_d = cap_msa.sort_values("total_m", ascending=False)["msa_name"].tolist()

SPLIT_COLORS = {
    "Direct (BPS, Greening, etc.)": "#4a7fab",
    "Rural Development":            TEAL,
}

bars_d = alt.Chart(long_d).mark_bar(height=12).encode(
    y=alt.Y("msa_name:N", sort=sort_d, title=None,
            axis=alt.Axis(labelFontSize=10, grid=False)),
    x=alt.X("amount_m:Q", title="CAP payments (£m, 2024)",
            stack="zero", axis=alt.Axis(grid=False)),
    color=alt.Color("payment_type:N",
                    scale=alt.Scale(
                        domain=list(SPLIT_COLORS.keys()),
                        range=list(SPLIT_COLORS.values())
                    ),
                    legend=alt.Legend(title=None, orient="top")),
    tooltip=[
        alt.Tooltip("msa_name:N",     title="MSA"),
        alt.Tooltip("payment_type:N", title="Type"),
        alt.Tooltip("amount_m:Q",     title="Amount (£m)", format=".1f"),
    ],
)

chart_d = bars_d.properties(
    width=420, height=420,
    title=alt.TitleParams(
        text="Direct vs Rural Development payments by MSA, 2024",
        subtitle="Direct payments dominate for all MSAs (BPS and successor schemes). Rural dev share varies.",
        fontSize=13, subtitleFontSize=9, anchor="start",
    ),
).configure_view(strokeWidth=0).configure(background=BACKGROUND).configure_axis(grid=False)

chart_d

## 6. Chart E — CAP payments per capita

In [ ]:
sort_e = cap_msa.dropna(subset=["gbp_per_capita"]).sort_values(
    "gbp_per_capita", ascending=False)["msa_name"].tolist()

bars_e = alt.Chart(cap_msa.dropna(subset=["gbp_per_capita"])).mark_bar(
    cornerRadiusEnd=3, height=14
).encode(
    y=alt.Y("msa_name:N", sort=sort_e, title=None,
            axis=alt.Axis(labelFontSize=10, grid=False)),
    x=alt.X("gbp_per_capita:Q", title="CAP payments per capita (£, 2024)",
            axis=alt.Axis(grid=False)),
    color=alt.Color("dot_color:N", scale=None),
    tooltip=[
        alt.Tooltip("msa_name:N",        title="MSA"),
        alt.Tooltip("gbp_per_capita:Q",  title="£ per head", format=".2f"),
        alt.Tooltip("total_m:Q",         title="Total (£m)", format=".1f"),
        alt.Tooltip("pop_2023:Q",        title="Population", format=","),
    ],
)
labels_e = alt.Chart(cap_msa.dropna(subset=["gbp_per_capita"])).mark_text(
    align="left", dx=4, fontSize=9, color=NAVY,
).encode(
    y=alt.Y("msa_name:N", sort=sort_e),
    x=alt.X("gbp_per_capita:Q"),
    text=alt.Text("gbp_per_capita:Q", format=".2f"),
)

chart_e = (bars_e + labels_e).properties(
    width=420, height=400,
    title=alt.TitleParams(
        text="CAP payments per capita by MSA, 2024",
        subtitle="Per-capita normalises for both area and population — shows agricultural subsidy burden/benefit per resident.",
        fontSize=13, subtitleFontSize=9, anchor="start",
    ),
).configure_view(strokeWidth=0).configure(background=BACKGROUND).configure_axis(grid=False)

chart_e